In [ ]:
# Import section
from pathlib import Path
from yt_dlp import YoutubeDL
import eyed3
from PIL import Image

In [ ]:
# Choose Song Area
musicLink = "https://www.youtube.com/watch?v=QrqjLoPbnyY"
TITLE = "Sneaky Snitch"
ARTIST = "Kevin MacLeod"

# Download Area
destination = Path(r"C:\REDACTED")  # Change this to your desired destination folder
ydl_opts = {
    'format': 'bestaudio/best',
    'writethumbnail': True,  # save thumbnail file next to output
    'outtmpl': str(destination / f"{TITLE}.%(ext)s"),
    'postprocessors': [{
        'key': 'FFmpegExtractAudio',
        'preferredcodec': 'mp3',
        'preferredquality': '320'
    }],
    # optional additions to reduce 403/forbidden errors
    'nocheckcertificate': True,
    'geo_bypass': True,
    'http_headers': {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36',
        'Referer': 'https://www.youtube.com/'
    }
}

# get thumbnail format
def getThumbnailFormat(info):
    # Find local thumbnail file that has same filename as TITLE but different extension
    for ext in ['jpg', 'png', 'webp']:
        thumbnail_path = destination / f"{TITLE}.{ext}"
        if thumbnail_path.exists():
            return ext

downloader = YoutubeDL(ydl_opts)
try:
    info = downloader.extract_info(musicLink, download=True)
except Exception as e:
    # surface helpful hints for 403 errors
    print('DownloadError:', e)
    print('If you see a 403 Forbidden: try updating yt-dlp, add a cookies file via --cookies, or supply http headers/user-agent.')
    raise
thumbnail_format = getThumbnailFormat(info)
thumbnail_location = destination / f"{TITLE}.{thumbnail_format}"

# Check if the thumbnail is a square image
with Image.open(thumbnail_location) as img:
    width, height = img.size
    # if the thumbnail is not square, we need to trim the longer side
    if width != height:
        min_dimension = min(width, height)
        left = (width - min_dimension) / 2
        top = (height - min_dimension) / 2
        right = (width + min_dimension) / 2
        bottom = (height + min_dimension) / 2
        img_cropped = img.crop((left, top, right, bottom))
        img_cropped.save(thumbnail_location)
        
audio_path = destination / f"{TITLE}.mp3"
audio = eyed3.load(audio_path)
if audio is None:
    raise FileNotFoundError(f"MP3 file not found: {audio_path}")
# ensure there is a tag to write to
if audio.tag is None:
    audio.initTag()
audio.tag.title = TITLE
audio.tag.artist = ARTIST
# map common extensions to proper MIME subtypes
_mime_map = {'jpg': 'jpeg', 'jpeg': 'jpeg', 'png': 'png', 'webp': 'webp'}
_sub = _mime_map.get(thumbnail_format, thumbnail_format)
mime_type = f"image/{_sub}"
image_data = thumbnail_location.read_bytes()
audio.tag.images.set(3, image_data, mime_type)
# delete the local thumbnail file after embedding
thumbnail_location.unlink(missing_ok=True)
audio.tag.save()